In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: d:\Pytnon_scripts\start_vector
✅ Sys path: ['d:\\Pytnon_scripts\\start_vector', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\DLLs']...


In [2]:
from src_oop.core.my_gspread import GoogleTabs
from src_oop.jobs.annual_procurement_plan.config import delivery_calculation_china, annual_procurement_plan
from src_oop.jobs.pivot_by_suppliers.config import pivot_by_suppliers_table
from src_oop.core.utils_general import clean_currency_value

import pandas as pd
from datetime import datetime
import gspread
import numpy as np

In [19]:
class PivotBySuppliers:
    # Определяю название таблицы и ее листов в классе
    pivot_by_suppliers_table_name = pivot_by_suppliers_table.get("title")
    sheet_simulator = pivot_by_suppliers_table.get("sheet_simulator")

    def __init__(self):
        # Создаем параметры подключения к сводной таблице поставщиков
        self._google_connect_to_pivot = None
        # Создаем параметры подключения к таблице Расчет поставки Китай_по обороту, к листу Заказы белые ТЕСТ
        self.china_table_from = delivery_calculation_china.get("title")
        self.white_orders_sheet_from = delivery_calculation_china.get("white_orders_sheet")
        self._google_connect_from_china = None
        # Колонки для фильтрации
        self.choosen_white_orders_columns = ["wild", "Модель", "Статус", "ФИН Статус", "Поставщик","Кол-во к заказу", "Сумма заказа, RMB", "Сумма аванса, RMB 🤖", "Сумма баланса, RMB 🤖", "Сумма баланса, RMB 2 🤖", "Итого долг"]


    def get_white_orders_data(self):
        # Подключаемся к листу Заказы белые ТЕСТ и получаем данные
        data = self.google_connect_from_china.sheet_title.get_all_values()
        data_from = data[4:] # Данные начиная с 5-й строки
        columns_from = data[3] # Названия колонок с 4-й строки
        df_full = pd.DataFrame(data_from, columns=columns_from)
        # Отфильтровываем только нужные колонки
        df = df_full[self.choosen_white_orders_columns]

        # Для отладки
        df = df[df["Поставщик"] == "TOPKO Product Group LTD."]


        # Очищаем цифровые колонки от лишних символов и приводим к числу
        dig_cols = ["Кол-во к заказу", "Сумма заказа, RMB", "Сумма аванса, RMB 🤖", "Сумма баланса, RMB 🤖", "Сумма баланса, RMB 2 🤖", "Итого долг"]
        for col in dig_cols:
            df[col] = df[col].apply(clean_currency_value)
        # Стираем полный датафрейм из оперативки
        del df_full
        return df
    
    # Задаем ленивое подключение к Свод_по_поставщикам, листу СВОД_тренажеры
    @property
    def google_connect_to_pivot(self):
        if self._google_connect_to_pivot is None:
            self._google_connect_to_pivot = GoogleTabs(PivotBySuppliers.pivot_by_suppliers_table_name, PivotBySuppliers.sheet_simulator)
        return self._google_connect_to_pivot
    # Задаем ленивое подключение к таблице Расчет поставки Китай_по обороту, к листу Заказы белые ТЕСТ    
    @property
    def google_connect_from_china(self):
        if self._google_connect_from_china is None:
            self._google_connect_from_china = GoogleTabs(self.china_table_from, self.white_orders_sheet_from)
        return self._google_connect_from_china


simulator = PivotBySuppliers()
df = simulator.get_white_orders_data()

✅ Успешное подключение к Расчет поставки Китай_по обороту -> Заказы белые ТЕСТ


In [20]:
df.head()

,wild,Модель,Статус,ФИН Статус,Поставщик,Кол-во к заказу,"Сумма заказа, RMB","Сумма аванса, RMB 🤖","Сумма баланса, RMB 🤖","Сумма баланса, RMB 2 🤖",Итого долг
30,wild1496,Велотренажер с дисплеем MN-5 (BYS-088B),прибыло,Оплачен полностью,TOPKO Product Group LTD.,3132.0,128412.0,12841.2,0.0,115570.8,0.0
31,wild1556,Виброплатформа для похудения 150 кг кардио сте...,прибыло,Оплачен полностью,TOPKO Product Group LTD.,2000.0,252800.0,25280.0,0.0,227520.0,0.0
32,wild1556,Виброплатформа для похудения 150 кг кардио сте...,прибыло,Оплачен полностью,TOPKO Product Group LTD.,2000.0,252800.0,25280.0,0.0,227520.0,0.0
33,wild1556,Виброплатформа для похудения 150 кг кардио сте...,прибыло,Оплачен полностью,TOPKO Product Group LTD.,500.0,63200.0,6320.0,0.0,56880.0,0.0
34,wild1496,Велотренажер с дисплеем MN-5 (BYS-088B),прибыло,Оплачен полностью,TOPKO Product Group LTD.,1866.0,76506.0,7650.6,0.0,68855.4,0.0


In [ ]:
df_debt = df.groupby(["Поставщик", "Статус", "ФИН Статус"], as_index=False).agg(
    Сумма_заказа=("Сумма заказа, RMB", "sum"),
    Кол_товара=("Кол-во к заказу", "sum"),
    Аванс=("Сумма аванса, RMB 🤖", "sum"),
    Аванс2=("Сумма баланса, RMB 🤖", "sum"),
    Баланс=("Сумма баланса, RMB 2 🤖", "sum"),
    Итого_долг=("Итого долг", "sum"),
)
df[""]
df_debt.head(10)

,Поставщик,Статус,ФИН Статус,Сумма_заказа,Кол_товара,Аванс,Аванс2,Баланс,Итого_долг
0,TOPKO Product Group LTD.,в производстве,ожидает оплаты баланса,213000.0,7500.0,42600.00,0.00,170400.00,170400.00
1,TOPKO Product Group LTD.,в пути,Оплачен полностью,0.0,180.0,0.00,0.00,0.00,0.00
2,TOPKO Product Group LTD.,в пути,ожидает оплаты баланса,3092522.4,31611.0,322159.04,296345.44,2474017.92,2474017.92
3,TOPKO Product Group LTD.,ждет оплаты аванса,просрочен аванс,10640029.6,217180.0,2128005.92,0.00,8512023.68,10640029.60
4,TOPKO Product Group LTD.,отмена,отменено,6743488.4,61418.0,0.00,0.00,6743488.40,6743488.40
5,TOPKO Product Group LTD.,прибыло,Оплачен полностью,2371247.2,24424.0,237124.72,0.00,2134122.48,0.00
6,TOPKO Product Group LTD.,прибыло,ожидает оплаты баланса,744915.2,6464.0,148983.04,0.00,595932.16,595932.16
7,TOPKO Product Group LTD.,прибыло,просрочен аванс,463056.0,7540.0,46305.60,46305.60,370444.80,416750.40
8,TOPKO Product Group LTD.,товар готов к вывозу,ожидает оплаты баланса,901420.0,11484.0,142097.20,38186.80,721136.00,721136.00
